[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-07-TemporalCNN <<](./QC-Py-Cloud-07-TemporalCNN.ipynb)

# Value Factor Z-Score — Selection multi-facteurs fondamentaux

**Source** : Projet etudiant ESGF 2026 (QC `31932810`), anonymise et adapte.
**Type** : `doc cloud` — resultats obtenus sur [QuantConnect Cloud](https://www.quantconnect.com/lab).

---

## Objectifs d’apprentissage

1. Comprendre le **Z-Score factoriel** : normaliser des fondamentaux hétérogènes en un score composite.
2. Distinguer **8 facteurs fondamentaux** et leur sens économique (value vs growth, qualité vs rendement).
3. Analyser pourquoi une stratégie value peut **sous-performer** pendant une décennie growth-dominée (2015-2024).
4. Interpréter les limites d’un backtest : Sharpe faible, PSR < 1%, drawdown élevé.

**Prérequis** : Concepts de value investing, ratios financiers (P/E, ROE, FCF yield).

**Durée estimée** : 30 min

## 1. Concept : Le Z-Score multi-facteurs fondamental

L’idée est simple : au lieu de trier les actions sur un seul critère (ex: P/E le plus bas), on combine **plusieurs facteurs fondamentaux** en un score unique.

### Les 8 facteurs

| Facteur | chemin fondamental | Sens | Direction favorisée |
|---------|-------------------|------|---------------------|
| Revenue Growth | `operation_ratios.revenue_growth` | Croissance du chiffre d’affaires | Supérieur = mieux |
| Gross Margin | `operation_ratios.gross_margin` | Marge brute | Supérieur = mieux |
| ROE | `operation_ratios.roe` | Retour sur capitaux propres | Supérieur = mieux |
| Operating Margin | `operation_ratios.operation_margin` | Marge opérationnelle | Supérieur = mieux |
| P/E Ratio | `valuation_ratios.pe_ratio` | Prix / bénéfices | **Inférieur = mieux** (value) |
| Book Yield | `valuation_ratios.book_value_yield` | Valeur comptable / prix | Supérieur = mieux |
| Earning Yield | `valuation_ratios.earning_yield` | Bénéfices / prix | Supérieur = mieux |
| FCF Yield | `valuation_ratios.fcf_yield` | Flux de trésorerie libres / prix | Supérieur = mieux |

### Méthode

1. **Sélection** : prendre les N actions les plus liquides (top 50 par dollar volume).
2. **Z-Score** : pour chaque facteur, calculer `(valeur - moyenne) / écart-type`.
3. **Inversion** : le P/E est inversé (`-z`) car un P/E bas = value.
4. **Composite** : moyenne des Z-Scores = score global.
5. **Sélection finale** : garder les top 10.
6. **Allocation** : équipondérée, rééquilibrage mensuel.

### Exercice 1 : Calcul du Z-Score composite

Le Z-Score normalise chaque facteur pour les rendre comparables. Pour un vecteur de valeurs $x_1, \ldots, x_n$ :

$$z_i = \frac{x_i - \mu}{\sigma}$$

Implémentez la fonction `zscore_composite` qui prend une matrice de facteurs (lignes = actifs, colonnes = facteurs) et retourne le score composite pour chaque actif.

In [1]:
# Exercice 1 : Calcul du Z-Score composite
# TODO etudiant : implémenter zscore_composite(factors_matrix, invert_cols=None)
# Indice : np.nanmean + np.nanstd sur axis=0, puis moyenne sur axis=1
# invert_cols : liste des indices de colonnes à inverser (ex: PE ratio)
def zscore_composite(factors_matrix, invert_cols=None):
    pass  # TODO etudiant : implémenter le Z-Score composite


## 2. Le piège du value investing (2015-2024)

La période 2015-2024 a été dominée par les **growth stocks** (FAANG/Mag7, tech, IA). Les facteurs value (P/E bas, book yield élevé) ont systématiquement sous-performé.

### Pourquoi ?

- **Taux bas** : l’argent gratuit favorise les entreprises en croissance rapide.
- **Concentration** : les Mag7 (Apple, Microsoft, Amazon, Google, Meta, Nvidia, Tesla) ont accaparé les rendements.
- **Intangibles** : les fondamentaux comptables traditionnels sous-évaluent les actifs incorporels (R&D, marques, réseaux).
- **Momentum croissance** : les actions growth ont bénéficié d’un cercle vertueux (appréciation → inflow → appréciation).

### Résultat : la stratégie value Z-Score sur 2015-2024

| Métrique | Valeur | Benchmark SPY (~) | Interprétation |
|----------|--------|--------------------|----------------|
| Sharpe | 0.227 | ~0.80-1.00 | **Sous-performance nette** |
| CAGR | 6.41% | ~15% | La moitié du marché |
| MaxDD | -36.5% | ~-25% (COVID) | Drawdown plus sévère |
| PSR | 0.8% | > 50% | **Non significatif** statistiquement |

> **Leçon** : une stratégie value à fondamentaux solides (8 facteurs diversifiés) peut tout de même sous-performer pendant une décennie entière. Le value n’est pas un signal universel — il est **régime-dépendant**.

### Exercice 2 : Identifier les facteurs défavorisés dans un contexte growth

Certains de nos 8 facteurs sont particulièrement pénalisés en période growth. Identifiez lesquels et pourquoi.

In [2]:
# Exercice 2 : Analyse des facteurs en contexte growth
# TODO etudiant : pour chaque facteur, indiquer si il est favorisé ou pénalisé
# en période growth (2015-2024) avec une courte justification
factors_analysis = {
    "revenue_growth": None,  # TODO etudiant : "favorisé" ou "pénalisé" + raison
    "gross_margin": None,
    "roe": None,
    "op_margin": None,
    "pe_ratio": None,
    "book_yield": None,
    "earning_yield": None,
    "fcf_yield": None,
}
# Indice : les facteurs de valorisation (P/E, book yield) pénalisent les growth stocks
print("Exercice a completer")

Exercice a completer


## 3. Le code QC Cloud

La stratégie `FundamentalZScoreAlgorithm` (projet QC `31932810`) :

```python
class FundamentalZScoreAlgorithm(QCAlgorithm):
    def initialize(self):
        self.set_start_date(2015, 1, 1)
        self.set_end_date(2024, 12, 31)
        self.set_cash(100_000)
        
        # 8 facteurs fondamentaux
        self._factor_names = [
            "revenue_growth", "gross_margin", "roe", "op_margin",
            "pe_ratio", "book_yield", "earning_yield", "fcf_yield",
        ]
        # Sélection : top 50 par liquidité, puis top 10 par Z-Score composite
        self._liquid_universe_size = 50
        self._final_universe_size = 10
```

Le cœur de la stratégie est dans `_select_assets` :
1. Tri par dollar_volume → top 50 liquides.
2. Extraction des 8 fondamentaux → matrice $50 \times 8$.
3. Z-Score par colonne (inversion du P/E).
4. Composite = moyenne des Z-Scores.
5. Top 10 → équipondérée, rééquilibrage mensuel.

## 4. Résultats QC Cloud

**Projet QC Cloud** : `31932810` ("Clone of Equity Aroon Trend" — nom trompeur, la classe est `FundamentalZScoreAlgorithm`).
**Période** : 2015-01-01 → 2024-12-31 (10 ans).

| Métrique | Valeur |
|----------|--------|
| **Sharpe** | 0.227 |
| **CAGR** | 6.41% |
| **MaxDD** | -36.5% |
| **PSR** | 0.8% |
| **Tradeable days** | 2516 |

### Interprétation

**Sharpe 0.227** : en dessous du seuil de rentabilité ajustée au risque. SPY a un Sharpe ~0.8-1.0 sur la même période. La stratégie ne compense pas le risque pris.

**CAGR 6.41%** : moins de la moitié du rendement du marché (~15%). En termes réels (après inflation), le rendement est marginal.

**MaxDD -36.5%** : pire que le benchmark. La diversification sur 10 actions value n’a pas protégé pendant les crises (COVID 2020, taux montants 2022).

**PSR 0.8%** : le Probabilistic Sharpe Ratio est proche de zéro. Cela signifie que le Sharpe observé (0.227) n’est **statistiquement pas distinguishable de zéro**. Le signal est du bruit, pas de l’alpha.

> **Conclusion** : la stratégie fonctionne en théorie (value factor académiquement valide), mais **échoue en pratique** sur cette décennie spécifique. C’est un cas d’école du risque de **régime dependency**.

### Exercice 3 : Proposer une amélioration régime-aware

Une stratégie value ne fonctionne pas dans tous les régimes de marché. Proposez un mécanisme simple pour désactiver le signal value quand le régime est défavorable.

In [3]:
# Exercice 3 : Amélioration régime-aware
# TODO etudiant : implémenter detect_growth_regime(spy_prices, lookback=252) -> bool
# Indice : comparer le rendement de SPY vs un proxy value (ex: VTV ETF)
# Si SPY >> VTV sur lookback jours, le régime est growth -> désactiver le signal value
def detect_growth_regime(spy_prices, lookback=252):
    return False  # TODO etudiant : implémenter la détection de régime

print("Exercice a completer")

Exercice a completer


## 5. Leçon principale : Le value n’est pas un signal universel

### Points clés

1. **Le value factor est régime-dépendant** : il fonctionne sur de longues périodes (Fama-French, 1992-2020) mais peut sous-performer pendant une décennie entière.

2. **Le Z-Score composite ne corrige pas le problème** : normaliser des fondamentaux ne change pas le fait que le marché préfère la croissance à la valeur comptable.

3. **Le PSR est un filtre essentiel** : un Sharpe de 0.227 avec PSR < 1% = signal non significatif. Toujours vérifier la **significativité statistique** avant de déployer.

4. **L’équipondérée sur 10 actions** concentre le risque : en période de stress, les corrélations montent et la diversification s’évapore.

5. **Les fondamentaux traditionnels** (P/E, book value) sous-évaluent les entreprises à actifs incorporels dominants (tech, plateforme, IA).

### Pour aller plus loin

- **Fama-French 5-factor model** : ajouter size, profitability, investment aux facteurs value et marché.
- **Regime detection** : utiliser le momentum relatif (growth vs value ETFs) comme filtre.
- **Dynamic weighting** : adapter l’allocation aux régimes détectés (value en régime value, growth en régime growth).

---

[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-07-TemporalCNN <<](./QC-Py-Cloud-07-TemporalCNN.ipynb)